# 01 - Data Cleaning

**Phase 1 of the project.** The PRD asks for missing-value handling,
categorical normalisation and duplicate removal. Both files turn out to be
structurally clean, so the real work is *type recovery*: two date columns and
one currency column arrive as free text.

Two questions have to be settled with evidence before anything downstream is
trustworthy:

1. Is `transaction_date` day-first or month-first?
2. Is `date_of_birth` day-first or month-first?


In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

from src import config as cfg
raw_clients = pd.read_csv(cfg.RAW_CLIENTS, dtype=str)
raw_props = pd.read_csv(cfg.RAW_PROPERTIES, dtype=str)
print("clients   ", raw_clients.shape)
print("properties", raw_props.shape)
raw_clients.head()


clients    (2000, 12)
properties (10000, 9)


,client_id,client_type,first_name,last_name,date_of_birth,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel
0,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,Yes,Website
1,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,No,Website
2,C0003,Individual,Kale,Gay,04-07-1959,M,USA,California,Home,4,Yes,Agency
3,C0004,Individual,Russell,Gross,11/25/1959,M,USA,California,Home,5,No,Website
4,C0005,Company,Marleez,Co,2/28/1976,M,USA,California,Investment,5,No,Website


## Structural checks: missing values, duplicates, referential integrity

In [2]:
print("clients missing cells :", int(raw_clients.isna().sum().sum()))
print("clients duplicate rows:", int(raw_clients.duplicated().sum()))
print("duplicate client_id   :", int(raw_clients['client_id'].duplicated().sum()))
print()
print("properties duplicate listing_id:", int(raw_props['listing_id'].duplicated().sum()))
print("properties missing client_ref  :", int(raw_props['client_ref'].isna().sum()))
print()
print("Missing client_ref against listing_status:")
print(pd.crosstab(raw_props['listing_status'], raw_props['client_ref'].isna()))


clients missing cells : 0
clients duplicate rows: 0
duplicate client_id   : 0

properties duplicate listing_id: 0
properties missing client_ref  : 2695

Missing client_ref against listing_status:
client_ref      False  True 
listing_status              
Available           0   2695
Sold             7305      0


The 2,695 missing `client_ref` values line up **exactly** with
`listing_status = Available`. They encode unsold inventory, not missing data,
so they must not be imputed - they are simply excluded from client-level
aggregation.


## Settling the date formats with evidence

In [3]:
import re
def shape(s):
    return re.sub(r"\d", "9", str(s))

print("date_of_birth serialisations:")
print(raw_clients['date_of_birth'].map(shape).value_counts().to_string())
print()
print("transaction_date serialisations:")
print(raw_props['transaction_date'].map(shape).value_counts().to_string())


date_of_birth serialisations:
date_of_birth
9/99/9999     871
99-99-9999    855
99/99/9999    274

transaction_date serialisations:
transaction_date
99-99-9999    10000


In [4]:
def parts(series):
    p = series.str.split(r"[-/]", regex=True)
    return (p.str[0].astype(int), p.str[1].astype(int))

# transaction_date: is the second component ever > 12?
a, b = parts(raw_props['transaction_date'])
print("transaction_date  part1 range", a.min(), "-", a.max(),
      "| part2 range", b.min(), "-", b.max())
print("=> part2 is always 1: every transaction is stamped to the 1st of a month.")
print("   Read month-first this gives 24 consecutive months (Jan-2024..Dec-2024..Dec-2025).")
print("   Read day-first it would give days 1-12 of January only, in two years - implausible.")
print()

dob = raw_clients['date_of_birth']
slash = dob[~dob.str.contains('-')]
a2, b2 = parts(slash)
print("date_of_birth, slash subset: part2 range", b2.min(), "-", b2.max())
print("=> the day component reaches 31, so the slash subset is provably MONTH-first.")
dash = dob[dob.str.contains('-')]
a3, b3 = parts(dash)
print("date_of_birth, dash subset : part1 max", a3.max(), "part2 max", b3.max())
print("=> both components <= 12: genuinely ambiguous, and unrecoverable.")


transaction_date  part1 range 1 - 12 | part2 range 1 - 1
=> part2 is always 1: every transaction is stamped to the 1st of a month.
   Read month-first this gives 24 consecutive months (Jan-2024..Dec-2024..Dec-2025).
   Read day-first it would give days 1-12 of January only, in two years - implausible.



date_of_birth, slash subset: part2 range 13 - 31
=> the day component reaches 31, so the slash subset is provably MONTH-first.
date_of_birth, dash subset : part1 max 12 part2 max 12
=> both components <= 12: genuinely ambiguous, and unrecoverable.


### Why the ambiguity does not matter

The two formats partition the data on `day <= 12` versus `day >= 13`: a
spreadsheet re-serialised every date whose day fitted in a month slot. We read
everything month-first, consistent with the two provably month-first columns.

Crucially, **age is measured at 31 December 2025**. Every client's birthday
has already passed by that date, so age collapses to `2025 - birth_year` and
the day/month ordering cannot influence it. The check below proves it.


In [5]:
d_month = pd.to_datetime(dob, format='mixed', dayfirst=False)
d_day   = pd.to_datetime(dob, format='mixed', dayfirst=True)
print("parsed dates that differ between the two readings:", int((d_month != d_day).sum()))
print("birth YEARS that differ                          :", int((d_month.dt.year != d_day.dt.year).sum()))
print("=> 790 dates differ, zero years differ, so age is unaffected.")


parsed dates that differ between the two readings: 790
birth YEARS that differ                          : 0
=> 790 dates differ, zero years differ, so age is unaffected.


## Run the packaged cleaner

In [6]:
from src.data_cleaning import load_and_clean
clients, properties, report = load_and_clean()
print(report.summary())


DATA CLEANING REPORT
clients      : 2000 in -> 2000 out
  missing cells      : 0
  duplicate rows     : 0
  duplicate client_id: 0
  DOB unparsed       : 0
  DOB day/month ambiguous rows: 855
properties   : 10000 in -> 10000 out
  duplicate rows     : 0
  duplicate listing_id: 0
  missing client_ref : 2695
  all missing refs are 'Available': True
  orphan client_ref  : 0
  sale_price unparsed: 0
  transaction window : 2024-01-01 .. 2025-12-01
linkage      : 2000 of 2000 clients have >=1 sold property
labels normalised (cells changed): 0
--------------------------------------------------------------------
note: Referential integrity holds: every client_ref resolves to a client_id.
note: All clients appear in the sold-property table, so behavioural features are available for the entire client base.
note: Missing client_ref values coincide exactly with listing_status='Available'; they encode unsold stock, not missing data, and are excluded from client aggregation rather than imputed.
note

In [7]:
properties[['listing_id','transaction_date','unit_category',
            'floor_area_sqft','sale_price','listing_status',
            'client_ref','price_per_sqft']].head()


,listing_id,transaction_date,unit_category,floor_area_sqft,sale_price,listing_status,client_ref,price_per_sqft
0,1012,2024-01-01,Apartment,1160.36,300385.62,Sold,C0027,258.872781
1,1015,2024-01-01,Apartment,782.25,208930.81,Sold,C0097,267.089562
2,1021,2024-01-01,Apartment,756.21,218585.92,Sold,C0113,289.054522
3,1030,2024-01-01,Apartment,743.09,246172.68,Sold,C0141,331.282456
4,2016,2024-01-01,Apartment,701.66,212265.67,Sold,C0146,302.519269
